# 随机森林（Random Forest）

随机森林（Breiman，2001）是机器学习中最常用、最强大的集成算法之一。

**核心思路**：用 Bootstrap 有放回抽样 + 随机特征选择，训练多棵决策树，再用**投票（分类）或平均（回归）**得到最终结果。

| 特性 | 说明 |
|------|------|
| 类型 | 集成学习 / Bagging |
| 基学习器 | 决策树（通常 CART） |
| 优势 | 抗过拟合、特征重要性、高准确率 |
| 核心超参数 | `n_estimators`、`max_features`、`max_depth` |

## 1. 集成学习基础

**为什么集成 > 单模型？**

假设有 $T$ 个独立分类器，每个错误率 $\epsilon < 0.5$，集成后错误率（多数投票）由二项分布给出：

$$P(\text{集成错误}) = \sum_{k=\lceil T/2 \rceil}^{T} \binom{T}{k} \epsilon^k (1-\epsilon)^{T-k}$$

例：$T=25$ 个分类器，$\epsilon=0.35$，集成错误率 $\approx 0.06$（远低于单个 0.35）。

**两大集成范式**：

| 方法 | 代表算法 | 训练方式 | 基学习器关系 |
|------|----------|----------|--------------|
| **Bagging** | 随机森林 | 并行，有放回采样 | 独立，低相关 |
| **Boosting** | XGBoost/AdaBoost | 串行，关注误分类 | 互补，逐步纠错 |

随机森林属于 Bagging，通过**降低方差**来提升性能。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb
from math import comb as c

# 可视化集成错误率 vs 单分类器错误率
epsilon_vals = np.linspace(0, 0.5, 200)
T_vals = [1, 5, 11, 25, 101]

plt.figure(figsize=(8, 5))
for T in T_vals:
    ensemble_err = []
    for eps in epsilon_vals:
        err = sum(c(T, k) * eps**k * (1-eps)**(T-k) for k in range(T//2 + 1, T+1))
        ensemble_err.append(err)
    plt.plot(epsilon_vals, ensemble_err, label=f'T={T}')

plt.plot([0, 0.5], [0, 0.5], 'k--', label='单分类器（基线）')
plt.xlabel('单分类器错误率 ε')
plt.ylabel('集成错误率')
plt.title('Bagging 集成：树数量 vs 错误率')
plt.legend()
plt.axvline(0.35, color='gray', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Bootstrap 采样 —— 数据多样性的来源

随机森林每棵树都在一个 **Bootstrap 样本**（有放回抽样）上训练：

- 从 $n$ 个样本中有放回地抽取 $n$ 次
- 约有 $1 - (1-\frac{1}{n})^n \approx 1 - e^{-1} \approx 63.2\%$ 的样本被选中
- 剩余约 **36.8% 的样本（OOB：Out-of-Bag）** 未被选中，可用作验证集

**OOB 误差**：用每棵树的 OOB 样本做验证，平均后得到无偏误差估计，**无需额外划分验证集**。

In [ ]:
np.random.seed(42)
n = 100
n_trees = 500
oob_counts = np.zeros(n)   # 每个样本被选为OOB的次数

for _ in range(n_trees):
    selected = np.random.choice(n, size=n, replace=True)
    oob_mask = np.ones(n, dtype=bool)
    oob_mask[selected] = False
    oob_counts += oob_mask

# 理论值：每棵树约36.8%的样本在OOB中
avg_oob_rate = oob_counts.sum() / (n * n_trees)
print(f"实验 OOB 比例: {avg_oob_rate:.4f}（理论值 ≈ {1-np.e**-1:.4f}）")

# 可视化：各样本被选入OOB的次数分布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(range(n), np.sort(oob_counts)[::-1], color='steelblue', alpha=0.7)
ax1.axhline(n_trees * (1-np.e**-1), color='red', linestyle='--', label=f'理论期望 ≈ {n_trees*(1-np.e**-1):.0f}')
ax1.set_xlabel('样本编号（按OOB次数排序）')
ax1.set_ylabel('被选为 OOB 的次数')
ax1.set_title(f'各样本 OOB 次数（{n_trees} 棵树）')
ax1.legend()

# n 变化时 OOB 比例收敛
ns = [5, 10, 20, 50, 100, 500, 1000]
rates = [1 - (1 - 1/ni)**ni for ni in ns]
ax2.semilogx(ns, rates, 'o-', color='tomato')
ax2.axhline(1 - np.e**-1, linestyle='--', color='gray', label=f'极限 {1-np.e**-1:.4f}')
ax2.set_xlabel('样本量 n（对数轴）')
ax2.set_ylabel('OOB 理论比例')
ax2.set_title('OOB 比例随样本量的收敛')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. 随机特征选择 —— 树间去相关

普通 Bagging 的问题：如果有一个极强的特征，每棵树都会首先选它，导致各树高度相关，集成效果打折。

随机森林在每个节点分裂时，**只从随机抽取的 $m$ 个特征中选最优分裂**：

$$m = \begin{cases} \sqrt{p} & \text{分类（默认）} \\ p/3 & \text{回归（默认）} \end{cases}$$

其中 $p$ 是总特征数。

**效果**：降低树之间的相关性 → 集成方差更小 → 泛化能力更强。

集成预测的方差（$T$ 棵树，树间相关系数 $\rho$，单树方差 $\sigma^2$）：

$$\text{Var}\left(\frac{1}{T}\sum_{t=1}^T h_t\right) = \rho \sigma^2 + \frac{1-\rho}{T}\sigma^2$$

- $T \to \infty$ 时：$\text{Var} \to \rho\sigma^2$（不能消除相关性带来的方差）
- 随机特征选择降低 $\rho$ → 集成效果更好

## 4. 随机森林算法流程

**训练阶段**（for $t = 1, \ldots, T$）：
1. Bootstrap 抽样：从训练集 $\mathcal{D}$ 有放回采样得到 $\mathcal{D}_t$
2. 在 $\mathcal{D}_t$ 上训练一棵决策树 $h_t$，每次节点分裂时随机选 $m$ 个特征

**预测阶段**：
- **分类**：多数投票 $\hat{y} = \text{mode}\{h_1(x), h_2(x), \ldots, h_T(x)\}$
- **回归**：取平均 $\hat{y} = \frac{1}{T}\sum_{t=1}^T h_t(x)$

In [ ]:
from sklearn.tree import DecisionTreeClassifier

class SimpleRandomForest:
    """简化版随机森林（仅用于演示核心逻辑）"""
    def __init__(self, n_estimators=10, max_features='sqrt', max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.random_state = random_state
        self.trees = []
        self.oob_indices = []

    def fit(self, X, y):
        n, p = X.shape
        rng = np.random.RandomState(self.random_state)
        m = int(np.sqrt(p)) if self.max_features == 'sqrt' else p

        for _ in range(self.n_estimators):
            # Bootstrap 采样
            idx = rng.choice(n, size=n, replace=True)
            oob_mask = np.ones(n, dtype=bool)
            oob_mask[idx] = False
            self.oob_indices.append(oob_mask)

            X_b, y_b = X[idx], y[idx]

            # 随机特征由 DecisionTreeClassifier 的 max_features 参数实现
            tree = DecisionTreeClassifier(
                max_features=m,
                max_depth=self.max_depth,
                random_state=rng.randint(1e6)
            )
            tree.fit(X_b, y_b)
            self.trees.append(tree)
        return self

    def predict(self, X):
        preds = np.array([t.predict(X) for t in self.trees])   # (T, n_samples)
        return np.apply_along_axis(
            lambda col: np.bincount(col.astype(int)).argmax(), axis=0, arr=preds
        )

    def oob_score(self, X, y):
        n = len(y)
        oob_preds = np.full((self.n_estimators, n), -1)
        for i, (tree, oob) in enumerate(zip(self.trees, self.oob_indices)):
            if oob.any():
                oob_preds[i, oob] = tree.predict(X[oob])

        correct = 0
        for j in range(n):
            votes = oob_preds[:, j]
            valid = votes[votes >= 0]
            if len(valid) > 0:
                pred = np.bincount(valid.astype(int)).argmax()
                correct += (pred == y[j])
        return correct / n


# 测试
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

rf = SimpleRandomForest(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X_tr_s, y_tr)

train_acc = np.mean(rf.predict(X_tr_s) == y_tr)
test_acc  = np.mean(rf.predict(X_te_s)  == y_te)
oob_acc   = rf.oob_score(X_tr_s, y_tr)

print(f"手写随机森林 — 训练集: {train_acc:.4f}  测试集: {test_acc:.4f}  OOB: {oob_acc:.4f}")

## 5. sklearn RandomForestClassifier —— 与单棵决策树对比

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 对比单棵决策树 vs 随机森林（不同深度）
depths = [None, 3, 5, 10]
results = {}

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_tr_s, y_tr)
    rf_sk = RandomForestClassifier(n_estimators=100, max_depth=depth,
                                    oob_score=True, random_state=42, n_jobs=-1)
    rf_sk.fit(X_tr_s, y_tr)

    results[f'DT(depth={depth})'] = {
        'train': accuracy_score(y_tr, dt.predict(X_tr_s)),
        'test':  accuracy_score(y_te, dt.predict(X_te_s)),
        'oob':   None
    }
    results[f'RF(depth={depth})'] = {
        'train': accuracy_score(y_tr, rf_sk.predict(X_tr_s)),
        'test':  accuracy_score(y_te, rf_sk.predict(X_te_s)),
        'oob':   rf_sk.oob_score_
    }

print(f"{'模型':<22} {'训练集':>8} {'测试集':>8} {'OOB':>8}")
print('-' * 50)
for name, r in results.items():
    oob_str = f"{r['oob']:.4f}" if r['oob'] else '   —  '
    print(f"{name:<22} {r['train']:>8.4f} {r['test']:>8.4f} {oob_str:>8}")

In [ ]:
from sklearn.datasets import make_moons
from sklearn.inspection import DecisionBoundaryDisplay

# 2D 数据可视化决策边界：单棵深树 vs 随机森林
X2d, y2d = make_moons(n_samples=300, noise=0.25, random_state=42)
X2d_tr, X2d_te, y2d_tr, y2d_te = train_test_split(X2d, y2d, test_size=0.3, random_state=42)

models = {
    '决策树（无限深度）': DecisionTreeClassifier(random_state=42),
    '决策树（depth=3）': DecisionTreeClassifier(max_depth=3, random_state=42),
    '随机森林（100棵，depth=3）': RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    '随机森林（100棵，无限深度）': RandomForestClassifier(n_estimators=100, random_state=42),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X2d_tr, y2d_tr)
    acc = accuracy_score(y2d_te, model.predict(X2d_te))
    DecisionBoundaryDisplay.from_estimator(
        model, X2d, ax=ax, response_method='predict',
        cmap='RdBu', alpha=0.4
    )
    ax.scatter(X2d_tr[:, 0], X2d_tr[:, 1], c=y2d_tr, cmap='RdBu',
               edgecolors='k', s=25, label='训练')
    ax.scatter(X2d_te[:, 0], X2d_te[:, 1], c=y2d_te, cmap='RdBu',
               edgecolors='white', s=25, marker='s', label='测试')
    ax.set_title(f'{name}\n测试准确率: {acc:.3f}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('决策树 vs 随机森林决策边界对比', fontsize=12)
plt.tight_layout()
plt.show()

## 6. 特征重要性

随机森林可以计算每个特征对模型的贡献程度：

**基于不纯度的重要性（MDI）**：
$$\text{Importance}(j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{\text{节点}v \text{ 以特征} j \text{ 分裂}} p(v) \cdot \Delta \text{Gini}(v)$$

其中 $p(v)$ 是到达节点 $v$ 的样本比例，$\Delta \text{Gini}$ 是该分裂带来的不纯度下降。

**注意**：MDI 对高基数特征（取值多的特征）有偏，`permutation_importance` 更公平（但更慢）。

In [ ]:
from sklearn.inspection import permutation_importance

rf_fi = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_fi.fit(X_tr_s, y_tr)

feature_names = data.feature_names

# MDI 特征重要性
mdi_imp = rf_fi.feature_importances_
mdi_std = np.std([t.feature_importances_ for t in rf_fi.estimators_], axis=0)

# Permutation 特征重要性
perm_result = permutation_importance(rf_fi, X_te_s, y_te, n_repeats=10, random_state=42, n_jobs=-1)
perm_imp = perm_result.importances_mean
perm_std = perm_result.importances_std

# 取各自 Top-10
top10_mdi  = np.argsort(mdi_imp)[-10:][::-1]
top10_perm = np.argsort(perm_imp)[-10:][::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(range(10), mdi_imp[top10_mdi][::-1],
         xerr=mdi_std[top10_mdi][::-1], color='steelblue', alpha=0.8)
ax1.set_yticks(range(10))
ax1.set_yticklabels([feature_names[i] for i in top10_mdi[::-1]], fontsize=8)
ax1.set_title('MDI 特征重要性（Top 10）')
ax1.set_xlabel('不纯度下降（归一化）')

ax2.barh(range(10), perm_imp[top10_perm][::-1],
         xerr=perm_std[top10_perm][::-1], color='tomato', alpha=0.8)
ax2.set_yticks(range(10))
ax2.set_yticklabels([feature_names[i] for i in top10_perm[::-1]], fontsize=8)
ax2.set_title('Permutation 特征重要性（Top 10，测试集）')
ax2.set_xlabel('准确率下降')

plt.tight_layout()
plt.show()

## 7. 关键超参数分析

| 超参数 | 含义 | 调参建议 |
|--------|------|----------|
| `n_estimators` | 树的数量 | 越多越稳定，通常 100~500；不会过拟合 |
| `max_features` | 每次分裂随机选的特征数 | 分类用 `'sqrt'`；回归用 `'log2'` 或 `0.3~0.5` |
| `max_depth` | 树的最大深度 | `None`（完全生长）通常最佳；噪声多时可限制 |
| `min_samples_split` | 节点继续分裂的最小样本数 | 默认 2；增大可减少过拟合 |
| `min_samples_leaf` | 叶节点最小样本数 | 默认 1；增大可平滑噪声 |
| `max_samples` | Bootstrap 采样比例 | 默认 1.0（全量采样）；减小可加速 |

In [ ]:
from sklearn.model_selection import cross_val_score

# 超参数影响实验：n_estimators 和 max_features
n_est_range = [1, 5, 10, 20, 50, 100, 200]
mf_options  = ['sqrt', 'log2', 0.3, 0.5, 1.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# n_estimators 对 OOB 误差的影响
oob_errors = []
for n_est in n_est_range:
    rf_tmp = RandomForestClassifier(n_estimators=n_est, oob_score=True, random_state=42, n_jobs=-1)
    rf_tmp.fit(X_tr_s, y_tr)
    oob_errors.append(1 - rf_tmp.oob_score_)

ax1.plot(n_est_range, oob_errors, 'o-', color='steelblue')
ax1.set_xlabel('n_estimators（树的数量）')
ax1.set_ylabel('OOB 误差')
ax1.set_title('树数量 vs OOB 误差')
ax1.set_xscale('log')

# max_features 对交叉验证的影响
cv_means, cv_stds = [], []
for mf in mf_options:
    rf_tmp = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42, n_jobs=-1)
    scores = cross_val_score(rf_tmp, X_tr_s, y_tr, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

x_pos = range(len(mf_options))
ax2.bar(x_pos, cv_means, yerr=cv_stds, color='tomato', alpha=0.8, capsize=4)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([str(mf) for mf in mf_options])
ax2.set_xlabel('max_features')
ax2.set_ylabel('5折交叉验证准确率')
ax2.set_title('max_features 对准确率的影响')
ax2.set_ylim(0.9, 1.0)

plt.tight_layout()
plt.show()

## 8. 随机森林回归

回归任务中，每棵树输出连续值，最终预测 = **所有树的均值**。

$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} h_t(x)$$

评估指标：MSE（均方误差）、MAE（平均绝对误差）、$R^2$（决定系数）。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1D 回归演示
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 10, 200))
y_reg = np.sin(X_reg) + 0.3 * X_reg - 0.02 * X_reg**2 + np.random.randn(200) * 0.3

X_reg = X_reg.reshape(-1, 1)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

X_plot = np.linspace(0, 10, 500).reshape(-1, 1)

regressors = {
    '单棵决策树（无限深度）': DecisionTreeRegressor(random_state=42),
    '随机森林（100棵）': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    '随机森林（10棵）': RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model) in zip(axes, regressors.items()):
    model.fit(X_tr_r, y_tr_r)
    y_pred = model.predict(X_plot)
    r2 = r2_score(y_te_r, model.predict(X_te_r))
    rmse = np.sqrt(mean_squared_error(y_te_r, model.predict(X_te_r)))

    ax.scatter(X_tr_r, y_tr_r, s=10, alpha=0.5, color='gray', label='训练点')
    ax.plot(X_plot, y_pred, color='tomato', linewidth=2, label='预测')
    ax.set_title(f'{name}\nR²={r2:.3f}  RMSE={rmse:.3f}', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('随机森林回归：平滑效应演示', fontsize=12)
plt.tight_layout()
plt.show()

## 9. 偏差-方差分解

随机森林降低方差的机制可以用**偏差-方差分解**理解：

$$\text{期望误差} = \underbrace{\text{Bias}^2}_{\text{欠拟合}} + \underbrace{\text{Variance}}_{\text{过拟合}} + \text{噪声}$$

- **单棵深决策树**：偏差低（拟合好），方差高（对数据敏感）
- **随机森林**：偏差略增（因为限制了特征），但**方差大幅下降**（树间不相关）
- 净效果：通常比单棵树更优

In [ ]:
from sklearn.datasets import make_regression

# 用多次重采样估算偏差和方差
def bias_variance_decomp(model, X_full, y_full, n_runs=50, test_size=0.3, seed=0):
    rng = np.random.RandomState(seed)
    X_test = np.linspace(X_full.min(), X_full.max(), 100).reshape(-1, 1)
    preds = []
    for _ in range(n_runs):
        idx = rng.choice(len(X_full), size=int(len(X_full)*(1-test_size)), replace=False)
        model.fit(X_full[idx], y_full[idx])
        preds.append(model.predict(X_test))
    preds = np.array(preds)
    mean_pred = preds.mean(axis=0)
    variance = preds.var(axis=0).mean()
    true_y = np.sin(X_test.ravel()) + 0.3 * X_test.ravel() - 0.02 * X_test.ravel()**2
    bias_sq = ((mean_pred - true_y)**2).mean()
    return bias_sq, variance

models_bv = {
    '单棵决策树（depth=2）': DecisionTreeRegressor(max_depth=2),
    '单棵决策树（depth=None）': DecisionTreeRegressor(),
    'RF (T=5)': RandomForestRegressor(n_estimators=5, random_state=0),
    'RF (T=50)': RandomForestRegressor(n_estimators=50, random_state=0),
    'RF (T=200)': RandomForestRegressor(n_estimators=200, random_state=0),
}

print(f"{'模型':<28} {'偏差²':>10} {'方差':>10} {'偏差²+方差':>12}")
print('-' * 65)
for name, model in models_bv.items():
    b, v = bias_variance_decomp(model, X_reg.ravel(), y_reg, n_runs=30)
    print(f"{name:<28} {b:>10.4f} {v:>10.4f} {b+v:>12.4f}")

## 10. 随机森林 vs 其他算法综合对比

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

compare_models = {
    '逻辑回归':       LogisticRegression(max_iter=1000, random_state=42),
    'KNN (k=5)':      KNeighborsClassifier(n_neighbors=5),
    '决策树':         DecisionTreeClassifier(random_state=42),
    'SVM (RBF)':      SVC(kernel='rbf', random_state=42),
    '随机森林(100)':  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

cv_results = {}
for name, model in compare_models.items():
    scores = cross_val_score(model, X_tr_s, y_tr, cv=5, scoring='accuracy')
    cv_results[name] = scores

fig, ax = plt.subplots(figsize=(9, 5))
positions = range(len(cv_results))
boxes = ax.boxplot(list(cv_results.values()), patch_artist=True, notch=True)
colors = ['#AED6F1', '#A9DFBF', '#FAD7A0', '#F1948A', '#BB8FCE']
for patch, color in zip(boxes['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xticks(range(1, len(cv_results)+1))
ax.set_xticklabels(list(cv_results.keys()), rotation=15)
ax.set_ylabel('5折交叉验证准确率')
ax.set_title('多算法对比（乳腺癌数据集）')
ax.set_ylim(0.85, 1.02)
plt.tight_layout()
plt.show()

print(f"\n{'算法':<20} {'均值':>8} {'标准差':>8}")
print('-' * 40)
for name, scores in cv_results.items():
    print(f"{name:<20} {scores.mean():>8.4f} ±{scores.std():>6.4f}")

## 总结

| 知识点 | 核心内容 |
|--------|----------|
| 集成原理 | 多个弱学习器投票/平均 → 强学习器；误差率 $\to$ 指数级下降 |
| Bootstrap | 有放回采样，约 63.2% 被选中，36.8% 用作 OOB 验证 |
| 随机特征 | 每节点仅用 $\sqrt{p}$ 个特征，降低树间相关性 $\rho$ |
| 偏差-方差 | 随机森林大幅降低方差，偏差略增，净效果显著优于单棵树 |
| OOB 误差 | 无需额外验证集，是随机森林天然的泛化估计 |
| 特征重要性 | MDI（快，对高基数有偏）vs Permutation（慢，更公平） |
| 关键参数 | `n_estimators`↑ 更稳，`max_features` 控制多样性，`max_depth` 控制过拟合 |

**随机森林 vs Boosting（XGBoost）**：
- 随机森林：并行训练，更快，天然抗过拟合，超参调整简单
- XGBoost：串行纠错，通常精度更高，但需要更细致的调参
- 实践建议：先用随机森林作为强基线，再用 XGBoost 冲刺精度